# Generating a lognormal microstructure using SynthetMic

In this notebook we give an example of how to use [SynthetMic](https://github.com/synthetic-microstructures/synthetmic) to generate a 3D microstructure (Laguerre diagram) with 10,000 grains, where the volumes of the grains are drawn from a lognormal distribution. In particular, we reproduce Example 5.5 from [this paper](https://www.tandfonline.com/doi/full/10.1080/14786435.2020.1790053). Moreover, we show how to mesh the resulting diagram, i.e., for every voxel in a cubic mesh, we compute which grain it belongs to. Note that the SynthetMic library can be installed via pip as follows: `pip install synthetmic`.

First we import the relevant libraries:

In [1]:
import numpy as np
from synthetmic import LaguerreDiagramGenerator
from synthetmic.plot import plot_cells_as_pyvista_fig
from synthetmic.utils import mesh_diagram
import time

Next we define the 3D box $\{ (x,y,z) \,: \, 0 \le x \le L_1, \; 0 \le y \le L_2, \; 0 \le z \le L_3 \}$:

In [2]:
L1 = 2. # length of the box in the x-direction 
L2 = 2. # length of the box in the y-direction
L3 = 2. # length of the box in the z-direction
box = np.array([[0, L1],[0, L2],[0, L3]])

We fix the number of grains (Laguerre cells):

In [3]:
# Number of grains
N = 10_000

The target volumes of the grains are drawn from a lognormal distribution. To be precise, we draw radii from a lognormal distribution and then convert these into equivalent volumes.

In [4]:
# Target volumes of the grains, drawn from a lognormal distribution

ln_mean = 1 # mean of the lognormal distribution
std_dev = 0.35 # standard deviation of the lognormal distribution

# Lognormal parameters (see here https://en.wikipedia.org/wiki/Log-normal_distribution):
Sigma = np.sqrt(np.log(1+(std_dev/ln_mean)**2))
Mu = -0.5*Sigma**2+np.log(ln_mean)

radii = np.random.lognormal(Mu,Sigma,N) # Draw N radii from a lognormal distribution
target_vols = 4/3*np.pi*radii**3 # Compute the corresponding sphere volumes
target_vols  = target_vols*L1*L2*L3/np.sum(target_vols) # Normalise the volumes so that the total volume equals the volume of the box

# Remark: Only the ratio std_dev/ln_mean matters here. E.g., taking (ln_mean,std_dev)=(1,0.35) gives exactly the same result (up to
# machine precision) as taking (ln_mean,std_dev)=(c,0.35*c) for any constant c. This is because the volumes of the grains are normalised.

Now we generate a 3D Laguerre diagram with 10,000 grains using Algorithm 2 from [this paper](https://www.tandfonline.com/doi/full/10.1080/14786435.2020.1790053). The volumes of the grains agree with the target volumes to within the specified tolerance of 1%.

In [5]:
percent_tol = 1. # Volume tolerance: maximum percentage error of the volumes of the grains
num_Lloyd = 5 # Number of Lloyd iterations. The Lloyd iterations regularise the microstructure
seeds = np.random.rand(N,3)@np.diag([L1,L2,L3]) # Initial seed locations (drawn at random from the uniform distribution on the box)

# Initialise a LaguerreDiagramGenerator object
diagram = LaguerreDiagramGenerator(tol=percent_tol, n_iter=num_Lloyd) 

# Call the fit method to generate a Laguerre diagram with grains of volumes target_vols (to within the specified tolerance percent_tol)
start = time.time()
diagram.fit(
        seeds=seeds,
        volumes=target_vols,
        domain=box,
)
end = time.time()
print(f'Run time = {end-start} seconds') 

iteration: 1/5, max_percentage_error: 0.0069%, mean_percentage_error: 0.0000%
Resetting weights to init_weights
iteration: 2/5, max_percentage_error: 0.0019%, mean_percentage_error: 0.0000%
Resetting weights to init_weights
iteration: 3/5, max_percentage_error: 0.0003%, mean_percentage_error: 0.0000%
iteration: 4/5, max_percentage_error: 0.0140%, mean_percentage_error: 0.0000%
iteration: 5/5, max_percentage_error: 0.0428%, mean_percentage_error: 0.0000%
Run time = 25.191161394119263 seconds


We can plot the Laguerre diagram as follows:

In [6]:
pl = plot_cells_as_pyvista_fig(generator=diagram, colorby=diagram.get_fitted_volumes(), notebook=True)
pl.show()

Widget(value='<iframe src="http://localhost:38057/index.html?ui=P_0x7fa1c2314d70_0&reconnect=auto" class="pyvi…

## Meshing a diagram

We give an example of how to voxelise a Laguerre diagram. For every voxel in a cubic mesh, we compute which grain (Laguerre cell) it belongs to.

First we create a cubic mesh:

In [7]:
Nx = 100 # number of grid points in the x-direction
Ny = 100 # number of grid points in the y-direction
Nz = 100 # number of gird points in the z-direction
x = np.linspace(0, L1, Nx)
y = np.linspace(0, L2, Ny)
z = np.linspace(0, L3, Nz)
X, Y, Z = np.meshgrid(x, y, z)
x = np.reshape(X, Nx*Ny*Nz)
y = np.reshape(Y, Nx*Ny*Nz)
z = np.reshape(Z, Nx*Ny*Nz)
points = np.vstack((x, y, z)).T 

For each mesh point, we compute which grain it belongs to. SynthetMic provides two algorithms for doing this. The default option is the following:

In [8]:
start = time.time()
grain_indices = mesh_diagram(points=points, pd=diagram.optimal_transport_.pd)
end = time.time()
print(f'Run time = {end-start} seconds') 
print(grain_indices) # grain_indices[k] is the index of the grain containing mesh point k

Run time = 24.490264654159546 seconds
[8344 8344 8344 ... 4317 4317 4317]


For large examples (like this example, with 10,000 grains and 1,000,000 voxels), it is often faster to use parallel computation as follows:

In [9]:
start = time.time()
grain_indices = mesh_diagram(points=points, pd=diagram.optimal_transport_.pd, parallel=True)
end = time.time()
print(f'Run time = {end-start} seconds') 
print(grain_indices) # grain_indices[k] is the index of the grain containing mesh point k

Run time = 14.237481117248535 seconds
[8344 8344 8344 ... 4317 4317 4317]
